# LensIQ: deploy multi-model Roboflow detector to Model Serving

Downloads every Roboflow model the app exposes, packages them as a single
MLflow PyFunc that dispatches on a `model_id` row column, registers in
Unity Catalog, and creates/updates one Model Serving endpoint
(`lensiq-roboflow-detector` by default).

Payload shape (matches the AppKit `serving()` plugin invoke):

```json
{"dataframe_records": [{"image": "<b64>", "model_id": "roboflow-universe-projects/license-plate-recognition-rxg4e/13", "conf": 0.35}]}
```

Response shape (matches the existing YOLO endpoint so the AppKit server's
normalizer keeps working):

```json
{"predictions": [[{"label": "...", "class_id": 0, "confidence": 0.8, "bbox": [x1,y1,x2,y2]}]]}
```

In [ ]:
dbutils.widgets.text("catalog", "iot_dev")
dbutils.widgets.text("schema", "lensiq")
dbutils.widgets.text("registered_name", "lensiq_roboflow_detector")
dbutils.widgets.text("endpoint_name", "lensiq-roboflow-detector")
dbutils.widgets.text("api_key_scope", "reggie_pierce")
dbutils.widgets.text("api_key_secret", "ROBOFLOW_API_KEY")
# Comma-separated list of Roboflow model ids (workspace/project/version).
dbutils.widgets.text(
    "model_ids",
    ",".join([
        "roboflow-universe-projects/license-plate-recognition-rxg4e/13",
        "cv-6rgre/spills-ax5xv/2",
        "june-2023-wet-floor-sign/wet-floor-sign2/1",
        "takoyati/cigarette-vape-detection/14",
        "sensormatic/slip-and-fall/2",
    ]),
)

In [ ]:
%pip install -q 'inference[clip]==0.55.0' mlflow>=2.13 pillow numpy
dbutils.library.restartPython()

In [ ]:
import logging
import os
import shutil

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
LOG = logging.getLogger("deploy_roboflow")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
REGISTERED = f"{CATALOG}.{SCHEMA}.{dbutils.widgets.get('registered_name')}"
ENDPOINT = dbutils.widgets.get("endpoint_name")
API_KEY_SCOPE = dbutils.widgets.get("api_key_scope")
API_KEY_SECRET = dbutils.widgets.get("api_key_secret")
MODEL_IDS = [m.strip() for m in dbutils.widgets.get("model_ids").split(",") if m.strip()]
API_KEY = dbutils.secrets.get(scope=API_KEY_SCOPE, key=API_KEY_SECRET)

LOG.info("Deploying %d Roboflow models -> %s", len(MODEL_IDS), ENDPOINT)
for m in MODEL_IDS:
    LOG.info("  - %s", m)

## Pre-download every model into a single artifact directory

The Roboflow `inference` package caches models under `MODEL_CACHE_DIR`. We
point it at `/tmp/roboflow_cache`, call `get_model` for each id (which
fetches the weights), and ship the whole cache directory as a single MLflow
artifact so the served endpoint loads everything from disk without an API
key at runtime.

In [ ]:
CACHE_DIR = "/tmp/roboflow_cache"
os.environ["MODEL_CACHE_DIR"] = CACHE_DIR
os.environ["ROBOFLOW_API_KEY"] = API_KEY
os.environ["ONNXRUNTIME_EXECUTION_PROVIDERS"] = "[CPUExecutionProvider]"
if os.path.exists(CACHE_DIR):
    shutil.rmtree(CACHE_DIR)
os.makedirs(CACHE_DIR, exist_ok=True)

from inference import get_model
for mid in MODEL_IDS:
    LOG.info("Downloading %s", mid)
    _m = get_model(model_id=mid, api_key=API_KEY)
    LOG.info("  ready: %s (classes=%s)", mid, getattr(_m, "class_names", None) or list(getattr(_m, "id2label", {}).values())[:8])

LOG.info("Cache contents:")
for root, dirs, files in os.walk(CACHE_DIR):
    for f in files:
        p = os.path.join(root, f)
        LOG.info("  %s (%d bytes)", p, os.path.getsize(p))

## PyFunc wrapper

Loads every cached model into a dict at `load_context`, then dispatches by
`model_id` at predict time. Output rows match the YOLO endpoint shape so
the AppKit server's `_normalizeDatabricks` can parse them without change.

In [ ]:
import base64
import io
import json

import mlflow
import mlflow.pyfunc
import numpy as np
import pandas as pd
from mlflow.models import infer_signature
from PIL import Image


class RoboflowMultiDetector(mlflow.pyfunc.PythonModel):
    """Multi-model PyFunc that dispatches by `model_id`.

    Inputs (per row):
      - image:    base64-encoded JPEG/PNG (with or without `data:` prefix)
      - model_id: Roboflow id `workspace/project/version`
      - conf:     optional confidence threshold, default 0.35

    Output (per row): list of `{label, class_id, confidence, bbox}` with
    `bbox` as `[x1, y1, x2, y2]` corner coordinates so it matches the
    existing YOLO endpoint's contract.
    """

    def load_context(self, context):
        import os as _os
        _os.environ["MODEL_CACHE_DIR"] = context.artifacts["model_cache"]
        _os.environ["ONNXRUNTIME_EXECUTION_PROVIDERS"] = "[CPUExecutionProvider]"

        from inference import get_model
        # Allowed model ids are baked in via parameters so the served endpoint
        # can't be tricked into pulling an arbitrary Roboflow workspace.
        self._allowed = set(json.loads(context.model_config["model_ids"]))
        self._models = {}
        for mid in self._allowed:
            self._models[mid] = get_model(model_id=mid)

    def _run_one(self, image_b64, model_id, conf):
        if not image_b64 or not model_id:
            return []
        if model_id not in self._models:
            return [{"label": "error", "class_id": -1, "confidence": 0.0,
                     "bbox": [0, 0, 0, 0],
                     "error": f"unknown model_id: {model_id}"}]
        if isinstance(image_b64, str) and image_b64.startswith("data:"):
            image_b64 = image_b64.split(",", 1)[1]
        img = Image.open(io.BytesIO(base64.b64decode(image_b64))).convert("RGB")
        c = float(conf if conf is not None else 0.35)
        results = self._models[model_id].infer(np.array(img), confidence=c)
        # `inference` returns a list of ObjectDetectionInferenceResponse, one
        # per input image. Each has `.predictions` (center-coord boxes).
        out = []
        for resp in results:
            for p in getattr(resp, "predictions", []) or []:
                cx, cy, w, h = float(p.x), float(p.y), float(p.width), float(p.height)
                x1 = int(round(cx - w / 2))
                y1 = int(round(cy - h / 2))
                x2 = int(round(cx + w / 2))
                y2 = int(round(cy + h / 2))
                out.append({
                    "label": getattr(p, "class_name", None) or getattr(p, "class", "object"),
                    "class_id": int(getattr(p, "class_id", -1)),
                    "confidence": float(p.confidence),
                    "bbox": [x1, y1, x2, y2],
                })
        return out

    def predict(self, context, model_input, params=None):
        if hasattr(model_input, "to_dict"):
            rows = model_input.to_dict(orient="records")
        elif isinstance(model_input, dict):
            rows = [model_input]
        else:
            rows = list(model_input)
        return [self._run_one(r.get("image"), r.get("model_id"), r.get("conf")) for r in rows]

## Log + register

In [ ]:
_TINY_PNG_B64 = (
    "iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAYAAAAfFcSJAAAADUlEQVR42mP8/5+hHgAH"
    "ggJ/PchI7wAAAABJRU5ErkJggg=="
)

sample_input = pd.DataFrame([
    {"image": _TINY_PNG_B64, "model_id": MODEL_IDS[0], "conf": 0.35},
])
sample_output = [[]]
signature = infer_signature(sample_input, sample_output)

mlflow.set_registry_uri("databricks-uc")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

with mlflow.start_run(run_name="deploy_roboflow") as run:
    info = mlflow.pyfunc.log_model(
        artifact_path="model",
        python_model=RoboflowMultiDetector(),
        artifacts={"model_cache": CACHE_DIR},
        signature=signature,
        input_example=sample_input,
        registered_model_name=REGISTERED,
        model_config={"model_ids": json.dumps(MODEL_IDS)},
        pip_requirements=[
            "mlflow>=2.13",
            "inference[clip]==0.55.0",
            "pillow",
            "numpy<2",
        ],
    )
LOG.info("Logged model URI: %s", info.model_uri)

## Create / update the serving endpoint

Uses a `Small` scale-to-zero served entity. First cold start takes ~5-10 min
as the container builds and the model + ONNX runtime load.

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import EndpointCoreConfigInput, ServedEntityInput

client = mlflow.MlflowClient()
versions = client.search_model_versions(f"name='{REGISTERED}'")
latest_version = max(versions, key=lambda v: int(v.version)).version
LOG.info("Deploying %s version %s -> endpoint %s", REGISTERED, latest_version, ENDPOINT)

served = ServedEntityInput(
    entity_name=REGISTERED,
    entity_version=latest_version,
    workload_size="Small",
    scale_to_zero_enabled=True,
)

w = WorkspaceClient()
try:
    w.serving_endpoints.get(name=ENDPOINT)
    LOG.info("Endpoint exists; updating config")
    w.serving_endpoints.update_config(name=ENDPOINT, served_entities=[served])
except Exception:
    LOG.info("Endpoint not found; creating")
    w.serving_endpoints.create(
        name=ENDPOINT,
        config=EndpointCoreConfigInput(name=ENDPOINT, served_entities=[served]),
    )
LOG.info("Submitted deployment for %s; watch Serving UI for readiness.", ENDPOINT)